This notebook provides an end-to-end pipeline for:

1. **Date-Based Photo Organization:** Extracting EXIF dates or parsing file name patterns to structure raw facial skin photos into date-based directories.
2. **Google MediaPipe Face Landmark Classification:** Utilizing Google's `FaceLandmarker` model (478 facial landmarks) to compute the relative nose position against face width boundaries.
3. **Automatic Profile Sorting:** Classifying and copying/moving photos into orientation subfolders (`front`, `left`, `right`, or `unclassified`).
4. **Interactive Jupyter Visualizations:** Displaying debug visuals including the 478-point mesh, face oval contour, binary face oval mask, and key landmark markers.
5. **Precise Asymmetric Face Cropping:** Extracting face-centered bounding boxes with custom vertical alignment (head top expansion and minimal chin padding), generating binary face oval masks, and performing in-place batch cropping to a standard 3:4 aspect ratio across all dataset subdirectories.

In [ ]:
# IMPORTS

import os
import re
import gc
import sys
import csv
import random
import logging
import urllib.request

from pathlib import Path
from shutil import copy2, move
from datetime import datetime

import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp

import logging
import re
import shutil
from datetime import datetime
from pathlib import Path

from PIL import Image, ExifTags

# PATHS

PROJECT_ROOT = Path.cwd().parent

# Main project directory
BASE_DIR = PROJECT_ROOT / "images" / "zuza"

# Input and output directories
RAW_PHOTOS_DIR = BASE_DIR / "inzynierka"
DATED_PHOTOS_DIR = BASE_DIR / "dated_fotoes"
MAP_OUTPUT_DIR = BASE_DIR / "acne_maps"
LOG_DIR = BASE_DIR / "logs"

# Model and log files
MODEL_PATH = BASE_DIR / "face_landmarker.task"
CLASSIFICATION_LOG_PATH = BASE_DIR / "classification_log.csv"
PROFILE_LOG_PATH = LOG_DIR / "mediapipe_profile_detection.log"

# MediaPipe model download URL
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)


from PIL import Image, ExifTags
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("MediaPipe:", mp.__version__)

base_options = python.BaseOptions(
    model_asset_path=str(MODEL_PATH)
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)

face_landmarker = vision.FaceLandmarker.create_from_options(options)

print("Face Landmarker: OK")

# GENERAL SETTINGS

# Supported image file extensions
VALID_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp",
    ".heic", ".heif", ".tif", ".tiff"
}

# Photo classification folders
PROFILE_FOLDERS = ("front", "left", "right", "unclassified")

# File handling mode: "copy" or "move"
FILE_ACTION = "copy"


# FACE PROFILE CLASSIFICATION

# Thresholds used to classify face orientation
LEFT_THRESHOLD = 0.30
RIGHT_THRESHOLD = 0.70


# FACE CROPPING SETTINGS

# Additional space above the head, relative to face height
TOP_HEAD_EXPANSION = 0.40

# Additional space below the chin, relative to face height
CHIN_PADDING = 0.08

# Additional space on the left and right sides
SIDE_PADDING = 0.08

# Target image aspect ratio: width / height
TARGET_ASPECT_RATIO = 0.75


# ACNE MAP SETTINGS

# Face orientation used to generate acne maps
TARGET_SIDE = "front"

# Output image dimensions
MAP_WIDTH = 600
MAP_HEIGHT = 800

# Transparency of the face overlay
FACE_ALPHA = 0.055

# Heatmap blur intensity
HEATMAP_SIGMA = 12

# Heatmap transparency
HEATMAP_ALPHA = 0.78

# Minimum confidence threshold for YOLO predictions
YOLO_CONFIDENCE = 0.25


# Landmark indices defining the face oval
FACE_OVAL_INDICES = [
    10, 338, 297, 332, 284, 251, 389, 356, 454,
    323, 361, 288, 397, 365, 379, 378, 400, 377,
    152, 148, 176, 149, 150, 136, 172, 58, 132,
    93, 234, 127, 162, 21, 54, 103, 67, 109
]

# Set default plot dimensions
plt.rcParams["figure.figsize"] = (8, 8)

LOG_DIR.mkdir(parents=True, exist_ok=True)
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("photo_pipeline")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-7s | %(message)s",
        datefmt="%H:%M:%S",
    )

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    file_handler = logging.FileHandler(
        PROFILE_LOG_PATH,
        encoding="utf-8",
    )
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)


# 6. CONFIGURATION CHECK

print("Project root:", PROJECT_ROOT.resolve())
print("Base directory:", BASE_DIR.resolve())
print("Raw photos:", RAW_PHOTOS_DIR.resolve())
print("Dated photos:", DATED_PHOTOS_DIR.resolve())
print("Model:", MODEL_PATH.resolve())
print("Map output:", MAP_OUTPUT_DIR.resolve())


In [ ]:
def ensure_model_exists(model_path: Path, model_url: str):
    """Download Google MediaPipe Face Landmarker model if not present locally."""
    if not model_path.exists():
        logger.info(f"Downloading model task file to '{model_path}'...")
        model_path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(model_url, model_path)
        logger.info("Download completed successfully!")
    else:
        logger.info("MediaPipe Face Landmarker model task file found locally.")

ensure_model_exists(MODEL_PATH, MODEL_URL)

In [ ]:
#  STEP 1: DATE-BASED PHOTO SORTING WITH LIVE DIAGNOSTICS

# EXIF DATE EXTRACTION
def get_exif_date(path: Path) -> str | None:
    """Extract and validate the original photo date from EXIF metadata."""

    try:
        with Image.open(path) as img:
            exif = img.getexif()

            # DateTimeOriginal is usually stored in the EXIF IFD.
            exif_ifd = exif.get_ifd(ExifTags.IFD.Exif)

            date_tags = [
                (exif_ifd, "DateTimeOriginal"),
                (exif_ifd, "DateTimeDigitized"),
                (exif, "DateTime"),
            ]

            for metadata, tag_name in date_tags:
                for tag_id, value in metadata.items():
                    tag = ExifTags.TAGS.get(tag_id, tag_id)

                    if tag == tag_name and value:
                        try:
                            parsed = datetime.strptime(
                                str(value).strip()[:19],
                                "%Y:%m:%d %H:%M:%S"
                            )
                            date_str = parsed.strftime("%Y-%m-%d")

                            logger.info(
                                f"[EXIF] {path.name}: found {date_str} "
                                f"using {tag_name}"
                            )
                            return date_str

                        except ValueError:
                            logger.warning(
                                f"[EXIF] Invalid {tag_name} value "
                                f"for {path.name}: {value!r}"
                            )

    except Exception as exc:
        logger.debug(
            f"[EXIF] Could not read metadata from {path.name}: "
            f"{type(exc).__name__}: {exc}"
        )

    logger.info(f"[EXIF] No usable EXIF date: {path.name}")
    return None


# 3. DATE EXTRACTION FROM FILENAME
def get_date_from_filename(path: Path) -> str | None:
    """
    Extract dates from filenames, including:
    IMG_20251124_175602381.jpg
    IMG20260913221921.jpg
    2025-11-24_photo.jpg
    """

    filename = path.stem

    patterns = [
        # IMG_20251124_175602381
        r"IMG_(20\d{2})(\d{2})(\d{2})_",

        # IMG20260913221921 or 20260913221921
        r"(20\d{2})(\d{2})(\d{2})(?:\d{6})?",

        # 2025-11-24 or 2025_11_24
        r"(20\d{2})[-_](\d{2})[-_](\d{2})",
    ]

    for pattern in patterns:
        match = re.search(pattern, filename, re.IGNORECASE)

        if not match:
            continue

        year, month, day = match.group(1, 2, 3)

        try:
            parsed = datetime(
                int(year), int(month), int(day)
            )
            date_str = parsed.strftime("%Y-%m-%d")

            logger.info(
                f"[FILENAME] {path.name}: "
                f"matched pattern '{pattern}' -> {date_str}"
            )
            return date_str

        except ValueError:
            logger.warning(
                f"[FILENAME] Invalid date in {path.name}: "
                f"{year}-{month}-{day}"
            )

    logger.warning(
        f"[FILENAME] No recognized date pattern: {path.name}"
    )
    return None

# 4. SOURCE AND TARGET PATH VALIDATION
def organize_photos_by_date(source_dir: Path, target_dir: Path) -> None:
    """Recursively find photos and copy them into date-named folders."""

    source_dir = Path(source_dir).resolve()
    target_dir = Path(target_dir).resolve()

    logger.info("=" * 70)
    logger.info("STARTING RECURSIVE PHOTO SORTING")
    logger.info(f"Source: {source_dir}")
    logger.info(f"Target: {target_dir}")

    if not source_dir.exists() or not source_dir.is_dir():
        logger.error(f"Invalid source directory: {source_dir}")
        return

    if source_dir == target_dir or source_dir in target_dir.parents:
        # The target is inside the source; exclude it from scanning.
        logger.info("Target is inside source; target subtree will be excluded.")

    try:
        target_dir.mkdir(parents=True, exist_ok=True)

        photos = [
            p for p in source_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() in VALID_EXTENSIONS
            and target_dir not in p.resolve().parents
        ]

    except OSError:
        logger.exception("Failed to scan source directory.")
        return

    logger.info(f"Photos discovered recursively: {len(photos)}")

    if not photos:
        logger.warning("No supported photos found in source or subfolders.")
        return

    copied = 0
    skipped = 0
    failed = 0

    for index, photo in enumerate(photos, start=1):
        logger.info(f"[{index}/{len(photos)}] Processing: {photo}")

        try:
            date = get_exif_date(photo)

            if date is None:
                date = get_date_from_filename(photo)

            if date is None:
                logger.warning(f"SKIPPED - No date found: {photo.name}")
                skipped += 1
                continue

            date_folder = target_dir / date
            date_folder.mkdir(parents=True, exist_ok=True)

            destination = date_folder / photo.name

            # Avoid overwriting files with the same name.
            if destination.exists():
                if (
                    destination.stat().st_size == photo.stat().st_size
                ):
                    logger.info(
                        f"Already copied; skipping duplicate: {destination}"
                    )
                    skipped += 1
                    continue

                logger.warning(
                    f"Filename collision; adding source folder name: "
                    f"{photo.name}"
                )
                destination = (
                    date_folder /
                    f"{photo.parent.name}_{photo.name}"
                )

            shutil.copy2(photo, destination)

            if (
                destination.exists()
                and destination.stat().st_size == photo.stat().st_size
            ):
                logger.info(f"COPY SUCCESS: {photo} -> {destination}")
                copied += 1
            else:
                logger.error(f"COPY VERIFICATION FAILED: {destination}")
                failed += 1

        except Exception:
            logger.exception(f"FAILED TO PROCESS: {photo}")
            failed += 1

    logger.info("=" * 70)
    logger.info("SORTING SUMMARY")
    logger.info(f"Discovered: {len(photos)}")
    logger.info(f"Copied:     {copied}")
    logger.info(f"Skipped:    {skipped}")
    logger.info(f"Failed:     {failed}")
    logger.info(f"Output:     {target_dir}")
    logger.info("=" * 70)



# RUN


logger.info(f"Source directory: {RAW_PHOTOS_DIR.resolve()}")
logger.info(f"Target directory: {DATED_PHOTOS_DIR.resolve()}")

organize_photos_by_date(
    source_dir=RAW_PHOTOS_DIR,
    target_dir=DATED_PHOTOS_DIR
)


In [ ]:
# STEP 2: MEDIAPIPE FACE LANDMARKER CORE FUNCTIONS

def create_face_landmarker(model_path):
    """Initialize MediaPipe FaceLandmarker instance."""
    base_options = python.BaseOptions(model_asset_path=str(model_path))
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1
    )
    return vision.FaceLandmarker.create_from_options(options)


def classify_facial_profile(detector, image_path):
    """
    Classify face orientation into ('front', 'left', 'right', or 'unclassified')
    using relative nose tip location against facial boundaries.
    """
    image = cv2.imread(str(image_path))
    if image is None:
        return 'unclassified', 0.0

    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    detection_result = detector.detect(mp_image)
    if not detection_result.face_landmarks:
        return 'unclassified', 0.0

    landmarks = detection_result.face_landmarks[0]

    # Key landmark indices:
    # 1   -> Nose tip
    # 234 -> Leftmost facial edge
    # 454 -> Rightmost facial edge
    
    nose_x = landmarks[1].x
    left_side_x = landmarks[234].x
    right_side_x = landmarks[454].x

    face_width = right_side_x - left_side_x
    if face_width == 0:
        return 'unclassified', 0.0

    nose_relative_pos = (nose_x - left_side_x) / face_width
    print(f"DEBUG: Nose relative position = {nose_relative_pos:.3f}")

    if nose_relative_pos < LEFT_THRESHOLD:
        return 'left', nose_relative_pos
    elif nose_relative_pos > RIGHT_THRESHOLD:
        return 'right', nose_relative_pos
    else:
        return 'front', nose_relative_pos

In [ ]:
# STEP 3: INTERACTIVE JUPYTER VISUALIZATION PIPELINE

def show_step(image_bgr: np.ndarray, title: str = ""):
    """Display BGR image using Matplotlib inside Jupyter Notebooks."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure()
    plt.imshow(image_rgb)
    plt.title(title, fontsize=12)
    plt.axis("off")
    plt.show()


def run_visual_inspection(sample_image_path: Path):
    """Run step-by-step visual demonstration of MediaPipe classification."""
    image = cv2.imread(str(sample_image_path))
    if image is None:
        print(f"Failed to read image for visualization: {sample_image_path}")
        return

    show_step(image, "STEP 1: Raw Input Image")

    # Step 2: MediaPipe Landmarks Detection
    detector = create_face_landmarker(MODEL_PATH)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
    detection_result = detector.detect(mp_image)

    if not detection_result.face_landmarks:
        print("NO FACE DETECTED!")
        return

    landmarks = detection_result.face_landmarks[0]
    h, w, _ = image.shape

    # Step 3: All 478 Mesh Points Visualization
    img_all_landmarks = image.copy()
    for lm in landmarks:
        cx, cy = int(lm.x * w), int(lm.y * h)
        cv2.circle(img_all_landmarks, (cx, cy), 1, (0, 255, 0), -1)
    show_step(img_all_landmarks, "STEP 2: Full 478 Face Mesh Landmarks")

    # Step 4: Contour & Key Points
    img_contour = image.copy()
    num_pts = len(FACE_OVAL_INDICES)
    for i in range(num_pts):
        idx1 = FACE_OVAL_INDICES[i]
        idx2 = FACE_OVAL_INDICES[(i + 1) % num_pts]
        pt1 = (int(landmarks[idx1].x * w), int(landmarks[idx1].y * h))
        pt2 = (int(landmarks[idx2].x * w), int(landmarks[idx2].y * h))
        cv2.line(img_contour, pt1, pt2, (0, 255, 255), 2)

    nose_lm, left_lm, right_lm = landmarks[1], landmarks[234], landmarks[454]
    pt_nose = (int(nose_lm.x * w), int(nose_lm.y * h))
    pt_left = (int(left_lm.x * w), int(left_lm.y * h))
    pt_right = (int(right_lm.x * w), int(right_lm.y * h))

    cv2.line(img_contour, pt_left, pt_right, (255, 255, 0), 1)
    cv2.circle(img_contour, pt_left, 6, (255, 0, 0), -1)   # Left edge (Blue)
    cv2.circle(img_contour, pt_right, 6, (0, 0, 255), -1)  # Right edge (Red)
    cv2.circle(img_contour, pt_nose, 8, (0, 255, 0), -1)   # Nose tip (Green)
    show_step(img_contour, "STEP 3: Contour & Key Orientation Landmarks")

    # Step 5: Final Classification
    face_width = right_lm.x - left_lm.x
    relative_pos = (nose_lm.x - left_lm.x) / face_width if face_width != 0 else 0.5

    print(f"DEBUG: Relative nose position = {relative_pos:.3f}")

    if relative_pos < LEFT_THRESHOLD:
        orientation = "LEFT PROFILE"
    elif relative_pos > RIGHT_THRESHOLD:
        orientation = "RIGHT PROFILE"
    else:
        orientation = "FRONT"

    img_final = img_contour.copy()
    # cv2.putText(img_final, f"Result: {orientation}", (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    # cv2.putText(img_final, f"Nose Ratio: {relative_pos:.2f}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.rectangle(img_final, (10, 10), (520, 135), (255, 255, 255), -1)
    cv2.putText(img_final, f"Result: {orientation}", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(img_final, f"Nose Ratio: {relative_pos:.2f}", (20, 115), cv2.FONT_HERSHEY_SIMPLEX, 1.05, (0, 0, 0), 3, cv2.LINE_AA)
    show_step(img_final, "STEP 4: Final Classification Overlay")


import random 

sample_photos = [p for p in DATED_PHOTOS_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS]
sample_photo = random.choice(sample_photos) if sample_photos else None
if sample_photo:
    print(f"Randomly selected photo: {sample_photo.name}")
    run_visual_inspection(sample_photo)
else:
    print(f"No images found in: {DATED_PHOTOS_DIR.resolve()}")

In [ ]:
# STEP 4: BATCH PROCESSOR & SUMMARY TABLE GENERATOR

def process_and_sort_dataset():
    """Iterate through date subfolders, classify photos, move/copy to profile folders, and print summary."""
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"MediaPipe model task not found at: {MODEL_PATH}")

    if not DATED_PHOTOS_DIR.exists():
        raise FileNotFoundError(f"Dataset directory not found at: {DATED_PHOTOS_DIR}")

    detector = create_face_landmarker(MODEL_PATH)
    summary = {}
    detailed_log = []

    day_folders = sorted([d for d in DATED_PHOTOS_DIR.iterdir() if d.is_dir()])
    logger.info("Starting batch classification processing...")

    for day_folder in day_folders:
        day_name = day_folder.name
        if day_name in {'front', 'left', 'right', 'unclassified'}:
            continue

        # Create destination directories per date
        subdirs = {
            'front': day_folder / 'front',
            'left': day_folder / 'left',
            'right': day_folder / 'right',
            'unclassified': day_folder / 'unclassified'
        }
        for path in subdirs.values():
            path.mkdir(exist_ok=True)

        summary[day_name] = {'front': 0, 'left': 0, 'right': 0, 'unclassified': 0}

        files = [
            f for f in day_folder.iterdir()
            if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
        ]

        for file_path in files:
            category, _ = classify_facial_profile(detector, file_path)
            summary[day_name][category] += 1
            detailed_log.append((day_name, file_path.name, category))

            target_path = subdirs[category] / file_path.name
            if FILE_ACTION == 'move':
                move(file_path, target_path)
            else:
                copy2(file_path, target_path)

In [ ]:
# STEP 4: BATCH PROCESSOR & SUMMARY TABLE GENERATOR

def process_and_sort_dataset():
    """Iterate through date subfolders, classify photos, move/copy to profile folders, and print summary."""
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"MediaPipe model task not found at: {MODEL_PATH}")

    if not DATED_PHOTOS_DIR.exists():
        raise FileNotFoundError(f"Dataset directory not found at: {DATED_PHOTOS_DIR}")

    detector = create_face_landmarker(MODEL_PATH)
    summary = {}
    detailed_log = []

    day_folders = sorted([d for d in DATED_PHOTOS_DIR.iterdir() if d.is_dir()])
    logger.info("Starting batch classification processing...")

    for day_folder in day_folders:
        day_name = day_folder.name
        if day_name in {'front', 'left', 'right', 'unclassified'}:
            continue

        # Create destination directories per date
        subdirs = {
            'front': day_folder / 'front',
            'left': day_folder / 'left',
            'right': day_folder / 'right',
            'unclassified': day_folder / 'unclassified'
        }
        for path in subdirs.values():
            path.mkdir(exist_ok=True)

        summary[day_name] = {'front': 0, 'left': 0, 'right': 0, 'unclassified': 0}

        files = [
            f for f in day_folder.iterdir()
            if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
        ]

        for file_path in files:
            category, _ = classify_facial_profile(detector, file_path)
            summary[day_name][category] += 1
            detailed_log.append((day_name, file_path.name, category))

            target_path = subdirs[category] / file_path.name
            if FILE_ACTION == 'move':
                move(file_path, target_path)
            else:
                copy2(file_path, target_path)


    # SUMMARY TABLE & CSV LOGGING

    print("=" * 75)
    print("DAILY FACIAL PROFILE CLASSIFICATION SUMMARY")
    print("=" * 75)
    header = f"{'Date (Folder)':<18} | {'Front':<8} | {'Left':<8} | {'Right':<8} | {'Unclassified':<12} | {'Total':<6}"
    print(header)
    print("-" * 75)

    tot_front, tot_left, tot_right, tot_unclass = 0, 0, 0, 0

    for day, counts in summary.items():
        day_total = sum(counts.values())
        tot_front += counts['front']
        tot_left += counts['left']
        tot_right += counts['right']
        tot_unclass += counts['unclassified']

        print(
            f"{day:<18} | {counts['front']:<8} | {counts['left']:<8} | "
            f"{counts['right']:<8} | {counts['unclassified']:<12} | {day_total:<6}"
        )

    print("-" * 75)
    grand_total = tot_front + tot_left + tot_right + tot_unclass
    print(
        f"{'TOTAL':<18} | {tot_front:<8} | {tot_left:<8} | "
        f"{tot_right:<8} | {tot_unclass:<12} | {grand_total:<6}"
    )
    print("=" * 75)

    # Save detailed CSV execution log
    csv_log_path = DATASET_DIR / "classification_log.csv"
    with open(csv_log_path, "w", encoding="utf-8") as f:
        f.write("Date_Folder,File_Name,Category\n")
        for day, filename, cat in detailed_log:
            f.write(f"{day},{filename},{cat}\n")

    logger.info(f"Detailed classification log saved to: {csv_log_path}")


# Run the complete batch processor
process_and_sort_dataset()

## PRECISE FACE CROPPING WITH MASK AND IMPROVED ASYMMETRY

In [ ]:
# # Config paths
# BASE_DIR = Path(r"../images/zuza")
# MODEL_PATH = BASE_DIR / "face_landmarker.task"
# DATED_PHOTOS_DIR = BASE_DIR / "dated_fotoes"

# print("Working directory:", Path.cwd())
# print("BASE_DIR:", BASE_DIR.resolve())
# print("DATED_PHOTOS_DIR:", DATED_PHOTOS_DIR.resolve())
# print("Folder exists:", DATED_PHOTOS_DIR.is_dir())
# print("Model exists:", MODEL_PATH.is_file())


# # Landmark indices forming the outer face oval in MediaPipe
# FACE_OVAL_INDICES = [
#     10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
#     397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136,
#     172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109
# ]


def create_landmarker(model_path: Path) -> vision.FaceLandmarker:
    """Initialize the MediaPipe Face Landmarker detector."""
    base_options = python.BaseOptions(model_asset_path=str(model_path))
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1
    )
    return vision.FaceLandmarker.create_from_options(options)


def process_face_crop_with_mask(
    image_path: Path,
    model_path: Path,
    top_head_expansion: float = 0.35,  # 35% of face height added to the top (for hair/top of head)
    chin_padding: float = 0.04,        # Only 4% of face height added below the chin
    side_padding: float = 0.08,        # Side margins
    target_aspect_ratio: float = 0.75  # 3:4 aspect ratio (width / height)
):
    """
    Detects the face, generates a binary face oval mask, and crops a precise 3:4 frame.
    """
    image = cv2.imread(str(image_path))
    if image is None:
        print(f"Error loading file: {image_path}")
        return None, None, None, None

    h, w, _ = image.shape
    detector = create_landmarker(model_path)

    # Landmark detection
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
    detection_result = detector.detect(mp_image)

    if not detection_result.face_landmarks:
        print(f"No face detected in: {image_path.name}")
        return None, None, None, None

    landmarks = detection_result.face_landmarks[0]


    # STEP 1: Generate Binary Mask for Face Oval

    mask = np.zeros((h, w), dtype=np.uint8)
    oval_points = []
    for idx in FACE_OVAL_INDICES:
        lm = landmarks[idx]
        px, py = int(lm.x * w), int(lm.y * h)
        oval_points.append([px, py])

    oval_pts_arr = np.array(oval_points, dtype=np.int32)
    cv2.fillPoly(mask, [oval_pts_arr], 255)

    # Image with semi-transparent red mask overlay (preview)
    masked_vis = image.copy()
    masked_vis[mask == 255] = cv2.addWeighted(
        masked_vis[mask == 255], 0.5, 
        np.full_like(masked_vis[mask == 255], (0, 0, 255)), 0.5, 0
    )


    # STEP 2: Precise Crop Calculation (Top/Bottom Asymmetry)

    x_coords = [lm.x * w for lm in landmarks]
    y_coords = [lm.y * h for lm in landmarks]

    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)

    face_w = x_max - x_min
    face_h = y_max - y_min

    # Dedicated margins for top (hair/head top) and chin
    crop_x1 = x_min - (face_w * side_padding)
    crop_x2 = x_max + (face_w * side_padding)
    crop_y1 = y_min - (face_h * top_head_expansion)  # Deep extension upward
    crop_y2 = y_max + (face_h * chin_padding)        # Minimal padding below chin

    crop_w = crop_x2 - crop_x1
    crop_h = crop_y2 - crop_y1

    # Enforce 3:4 vertical aspect ratio
    current_ratio = crop_w / crop_h

    if current_ratio < target_aspect_ratio:
        # Frame too narrow -> expand sideways
        new_w = crop_h * target_aspect_ratio
        diff_w = new_w - crop_w
        crop_x1 -= diff_w / 2
        crop_x2 += diff_w / 2
    else:
        # Frame too wide -> expand vertically
        new_h = crop_w / target_aspect_ratio
        diff_h = new_h - crop_h
        crop_y1 -= diff_h * 0.7  # 70% expansion upward
        crop_y2 += diff_h * 0.3  # 30% expansion downward

    # Image boundaries check
    x1 = int(max(0, crop_x1))
    x2 = int(min(w, crop_x2))
    y1 = int(max(0, crop_y1))
    y2 = int(min(h, crop_y2))

    cropped_face = image[y1:y2, x1:x2]

    return image, mask, masked_vis, cropped_face



# VISUALIZATION AND TEST

sample_photo = list(DATED_PHOTOS_DIR.glob("**/*.jpg"))[7] if list(DATED_PHOTOS_DIR.glob("**/*.jpg")) else None  # Select eighth image from dataset folder

    
if sample_photo and MODEL_PATH.exists():
    print(f"Testing on image: {sample_photo.name}")

    orig, mask, masked_vis, cropped = process_face_crop_with_mask(
        sample_photo, 
        MODEL_PATH,
        top_head_expansion=0.4,  # Raises top edge (hair/head top)
        chin_padding=0.08,        # Cuts excess neck/chin area
        target_aspect_ratio=0.75  # 3:4 aspect ratio
    )

    if cropped is not None:
        plt.figure(figsize=(16, 5))

        # 1. Original
        plt.subplot(1, 4, 1)
        plt.imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
        plt.title("1. Original")
        plt.axis("off")

        # 2. Binary Mask
        plt.subplot(1, 4, 2)
        plt.imshow(mask, cmap="gray")
        plt.title("2. Binary Face Oval Mask")
        plt.axis("off")

        # 3. Mask Overlay
        plt.subplot(1, 4, 3)
        plt.imshow(cv2.cvtColor(masked_vis, cv2.COLOR_BGR2RGB))
        plt.title("3. Detected Face Area Preview")
        plt.axis("off")

        # 4. Adjusted 3:4 Crop
        plt.subplot(1, 4, 4)
        plt.imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
        plt.title(f"4. Cropped Face (3:4)\n{cropped.shape[1]}x{cropped.shape[0]}px")
        plt.axis("off")

        plt.tight_layout()
        plt.show()
else:
    print("Please make sure MODEL_PATH and at least one image in DATED_PHOTOS_DIR exist.")

## BATCH PROCESSING: CROP ALL IMAGES IN-PLACE

In [ ]:
def batch_crop_dataset_in_place(
    dataset_dir: Path,
    model_path: Path,
    top_head_expansion: float = 0.35,
    chin_padding: float = 0.06,
    side_padding: float = 0.08,
    target_aspect_ratio: float = 0.75
):
    """
    Recursively iterates through dataset_dir and all subdirectories,
    crops every supported image using process_face_crop_with_mask,
    and overwrites the original file in-place.
    """
    if not dataset_dir.exists():
        print(f"Directory not found: {dataset_dir}")
        return

    # Find all images recursively
    all_image_paths = [
        p for p in dataset_dir.rglob("*") 
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
    ]

    total_files = len(all_image_paths)
    print(f"Found {total_files} images to process in '{dataset_dir}'.")

    successful_count = 0
    skipped_count = 0

    for idx, image_path in enumerate(all_image_paths, start=1):
        _, _, _, cropped_face = process_face_crop_with_mask(
            image_path=image_path,
            model_path=model_path,
            top_head_expansion=top_head_expansion,
            chin_padding=chin_padding,
            side_padding=side_padding,
            target_aspect_ratio=target_aspect_ratio
        )

        if cropped_face is not None and cropped_face.size > 0:
            # Overwrite original image in place
            cv2.imwrite(str(image_path), cropped_face)
            successful_count += 1
            print(f"[{idx}/{total_files}] Cropped and saved in-place: {image_path.relative_to(dataset_dir)}")
        else:
            skipped_count += 1
            print(f"[{idx}/{total_files}] Skipped (no face detected or read error): {image_path.relative_to(dataset_dir)}")

    print("\n" + "=" * 60)
    print("BATCH CROPPING SUMMARY")
    print("=" * 60)
    print(f"Total processed files : {total_files}")
    print(f"Successfully cropped  : {successful_count}")
    print(f"Skipped / Failed      : {skipped_count}")
    print("=" * 60)


# Execute batch cropping across DATASET_DIR and all subfolders

batch_crop_dataset_in_place(
    dataset_dir=DATED_PHOTOS_DIR,
    model_path=MODEL_PATH,
    top_head_expansion=0.4,
    chin_padding=0.08,
    side_padding=0.08,
    target_aspect_ratio=0.75
)